In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct


import openai

Use OPEN API Key for embeddings and generation

In [2]:
from dotenv import load_dotenv
import os
from pathlib import Path

# Load environment variables from .env file in the project root
env_path = Path("../../.env")  # Adjust path if needed
if env_path.exists():
    load_dotenv(env_path, override=True)
else:
    # Try alternative path
    env_path = Path(".env")
    if env_path.exists():
        load_dotenv(env_path, override=True)

# Initialize OpenAI client with API key
# Option 1: Using environment variable (recommended)
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Option 2: If you prefer to set it directly (less secure, but simple for testing)
# openai_client = openai.OpenAI(api_key="your-api-key-here")

# Verify the API key is set
if os.getenv("OPENAI_API_KEY"):
    print("✓ OpenAI API key loaded from environment")
else:
    print("⚠ Warning: OPENAI_API_KEY not found in environment variables")
    print("  Make sure your .env file contains: OPENAI_API_KEY=your-key-here")

✓ OpenAI API key loaded from environment


Embedding Function

In [5]:
def get_embedding(text, model="text-embedding-3-small"):
    # Use the openai_client instance created in Cell 1
    response = openai_client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

Retrieval Function

In [6]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

## qdrant_client.query_points does the ANN match between the query and the vectorised data stored in quadrant database
## results is a list of points
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,

        ## why deconstructing qdrant_client.query_points into lists ?
    }

In [ ]:
# ============================================
# EXPLANATION: Hashability - Tuples vs Lists
# ============================================

# ============================================
# EXAMPLE 1: Tuples as Dictionary Keys
# ============================================

# ✅ WORKS: Tuple as dictionary key (because tuples are immutable/hashable)
movie_ratings = {
    ("B07FN83ZK1", "Action"): 4.5,
    ("B08XYZ1234", "Comedy"): 4.2,
    ("B09ABC5678", "Drama"): 4.8,
}

print("✅ Example 1: Tuple as dictionary key")
print(f"   Rating for ('B07FN83ZK1', 'Action'): {movie_ratings[('B07FN83ZK1', 'Action')]}")
print(f"   All keys: {list(movie_ratings.keys())}")

# ❌ DOESN'T WORK: List as dictionary key (because lists are mutable/unhashable)
print("\n❌ Example 2: List as dictionary key (will fail)")
try:
    movie_ratings_bad = {
        ["B07FN83ZK1", "Action"]: 4.5,  # This will raise TypeError
    }
except TypeError as e:
    print(f"   Error: {e}")
    print("   Reason: Lists cannot be dictionary keys because they're mutable!")

# ============================================
# EXAMPLE 3: Tuples as Set Elements
# ============================================

# ✅ WORKS: Tuple in a set (because tuples are hashable)
unique_movies = {
    ("B07FN83ZK1", "Action"),
    ("B08XYZ1234", "Comedy"),
    ("B09ABC5678", "Drama"),
    ("B07FN83ZK1", "Action"),  # Duplicate - will be ignored
}

print("\n✅ Example 3: Tuples in a set")
print(f"   Unique movies (duplicates removed): {unique_movies}")
print(f"   Count: {len(unique_movies)}")  # Only 3, not 4

# ❌ DOESN'T WORK: List in a set (because lists are unhashable)
print("\n❌ Example 4: Lists in a set (will fail)")
try:
    unique_movies_bad = {
        ["B07FN83ZK1", "Action"],  # This will raise TypeError
        ["B08XYZ1234", "Comedy"],
    }
except TypeError as e:
    print(f"   Error: {e}")
    print("   Reason: Lists cannot be set elements because they're mutable!")

# ============================================
# EXAMPLE 5: When Tuples Are NOT Hashable
# ============================================

# ⚠️ CAUTION: Tuple containing unhashable elements (like lists) is NOT hashable
print("\n⚠️ Example 5: Tuple with unhashable elements")
try:
    bad_tuple = ([1, 2, 3], "test")  # Tuple contains a list
    bad_dict = {bad_tuple: "value"}  # This will fail!
except TypeError as e:
    print(f"   Error: {e}")
    print("   Reason: Tuple contains a list, making the tuple itself unhashable!")

# ✅ GOOD: Tuple with only hashable elements
good_tuple = (1, 2, "test", (3, 4))  # All elements are hashable
good_dict = {good_tuple: "value"}  # This works!
print(f"\n✅ Example 6: Tuple with hashable elements works!")
print(f"   Dictionary: {good_dict}")

# ============================================
# SUMMARY
# ============================================
print("\n" + "="*60)
print("SUMMARY:")
print("="*60)
print("✅ Tuples are hashable (can be dict keys, set elements)")
print("   - IF all their elements are hashable")
print("   - Because tuples are immutable")
print("\n❌ Lists are NOT hashable (cannot be dict keys, set elements)")
print("   - Because lists are mutable")
print("\n💡 Hashable types: int, float, str, tuple, frozenset")
print("💡 Unhashable types: list, dict, set")
print("="*60)

Format retreived context data so as to be fed into LLM for generation as a context

In [ ]:

def preprocess_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [ ]:
preprocessed_context = preprocess_context(retrieved_context)

Build prompt template and substitute components in the template that is to be pass to LLM for generation of answer

In [ ]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt